In [0]:
catalog = "dbr_dev_ua5816bd"
bronze_schema = "mialkovska_viktor594_bronze"
scope="kotenko-kv-scope"
event_hub_namespace = "evhua5816bd"
event_hub_name = "kotenko-viktoria-tristar"

bronze_table = f"{catalog}.{bronze_schema}.tristar_events"

checkpoint_path = (
    f"/Volumes/{catalog}/mialkovska_viktor594/raw_files/"
    "tristar_eventhub/checkpoint"
)
bronze_checkpoint_path = f"{checkpoint_path}/bronze"

In [0]:
event_hub_connection_str = dbutils.secrets.get(
    scope=scope,
    key="evh-connection-string"
)

print(bool(event_hub_connection_str))

In [0]:
bootstrap_servers = f"{event_hub_namespace}.servicebus.windows.net:9093"

sasl_config = (
    'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule '
    f'required username="$ConnectionString" '
    f'password="{event_hub_connection_str}";'
)

In [0]:
raw_df = (
    spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", bootstrap_servers)
        .option("subscribe", event_hub_name)
        .option("kafka.security.protocol", "SASL_SSL")
        .option("kafka.sasl.mechanism", "PLAIN")
        .option("kafka.sasl.jaas.config", sasl_config)
        .option("startingOffsets", "latest")
        .load()
)

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit

bronze_df = (
    raw_df
    .selectExpr(
        "CAST(value AS STRING) AS raw_payload",
        "partition",
        "offset",
        "timestamp AS eh_enqueued_timestamp"
    )
    .withColumn("ingestion_timestamp", current_timestamp())
)

In [0]:
query = (
    bronze_df.writeStream
    .format("delta")
    .option("checkpointLocation", bronze_checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

In [0]:
spark.table(bronze_table).count()